In [1]:
#we will be performing dimensionality reduction using PCA
import sys; sys.path.append("..")
import pandas as pd
import numpy as np
from src.data_loader import PROCESSED_DIR

In [2]:
X_train = pd.read_parquet(PROCESSED_DIR / "X_train.parquet")
y_train = pd.read_parquet(PROCESSED_DIR / "y_train.parquet")["readmitted_30"]
groups_train = pd.read_parquet(PROCESSED_DIR / "groups_train.parquet")["patient_nbr"]

In [ ]:
# PCA on numerical columns only: drop categoricals, all-NaN columns, and ID columns (ints but nominal)
id_cols = ["admission_type_id", "discharge_disposition_id", "admission_source_id"]
X_test = pd.read_parquet(PROCESSED_DIR / "X_test.parquet")
num_cols = [c for c in X_train.select_dtypes("number").columns
            if c not in id_cols and X_train[c].notna().any()]
X_train_num = X_train[num_cols]
X_test_num = X_test[num_cols]
print(num_cols)
print(X_train_num.shape, X_test_num.shape, X_train_num.isna().sum().sum())

In [3]:
def center(X):
    mu=X.mean(axis=0)
    return X-mu , mu

def covariance_matrix(x_center):
    n=x_center.shape[0]
    return (x_center.T @ x_center)/(n-1)

Xc,mu = center(X_train)
C= covariance_matrix(Xc)


                         race  gender  age  admission_type_id  \
encounter_id                                                    
149190              Caucasian     NaN    2                  1   
64410         AfricanAmerican     NaN    3                  1   
500364              Caucasian     NaN    4                  1   
16680               Caucasian     NaN    5                  1   
35754               Caucasian     NaN    6                  2   

              discharge_disposition_id  admission_source_id  time_in_hospital  \
encounter_id                                                                    
149190                               1                    7                 3   
64410                                1                    7                 2   
500364                               1                    7                 2   
16680                                1                    7                 1   
35754                                1                    

In [ ]:
def pca_fit(X,k):
    Xc, mu = center(X)
    C = covariance_matrix(Xc)
    vals, vecs = np.linalg.eigh(C)
    idx = np.argsort(vals)[::-1]
    vals = vals[idx]
    vecs = vecs[:, idx]

    vals_k = vals[:k]
    vecs_k = vecs[:, :k]
    evr = vals_k / np.sum(vals)

    return {
        'mean': mu,
        'components': vecs_k,
        'explained_variance': vals_k,
        'explained_variance_ratio': evr,
    }



In [ ]:
# standardize with train statistics so large-scale columns don't dominate
mu_s, sd_s = X_train_num.mean(), X_train_num.std()
X_std = (X_train_num - mu_s) / sd_s

model = pca_fit(X_std, k=5)
print(model['explained_variance_ratio'], model['explained_variance_ratio'].sum())